## quantify segmentation error: expert annotation 

In [ ]:
# --- repo path bootstrap (added by the port) ---
import sys, pathlib
ROOT = next(p for p in pathlib.Path.cwd().parents if (p / "utils" / "paths.py").is_file())
sys.path.insert(0, str(ROOT))
from utils.paths import (profiles, features, feature_output, figdir, metadata,
                         data_dir, external, require)
from utils.panels import save_panel

import pandas as pd
import glob
import numpy as np
import os
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

# Settings to display more columns and rows
pd.set_option("max_colwidth", 200)
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 10)

# Find the input directory using glob
input_dirs = glob.glob('../*expert*/')
if input_dirs:
	pass  # was os.chdir; the bootstrap above handles paths
else:
	print("Warning: No input directory found.")


#### Load annotations

In [ ]:
# 7.7 GB of expert + Cellpose masks, held outside the repo.
# Absent? the committed table below is what the panel plots.
base_dir = external('expert-annotation')
CACHED_IOU = ROOT / 'analysis' / '3_SupplFigure2' / 'data' / 'segmentation_iou_cached.csv'
HAVE_MASKS = base_dir.is_dir()
expert_dir = base_dir / 'expert_npy'
cellpose_dir = base_dir / 'cellpose_npy'

def load_all_data(expert_dir, cellpose_dir):

    data = {}

    for source, data_dir in [('expert', expert_dir), ('cellpose', cellpose_dir)]:
        files = list(Path(data_dir).glob('*.npy'))

        for f in files:
            parts = f.stem.split('_')
            barcode = parts[0]          # PB000137
            seg_type = parts[1]         # cell or nuclei
            well = parts[3]             # D12
            plane = parts[4]             # 1

            key = (barcode, seg_type, well, plane, source)
            data[key] = np.load(f, allow_pickle=True)

    return data

# Load all data
all_labels = load_all_data(expert_dir, cellpose_dir)
print(f"Loaded {len(all_labels)} masks")

# Show sample keys
sample_keys = list(all_labels.keys())[:5]
print("Sample keys:", sample_keys)

#### Report across depth bins, treatment class, channel, and cell line

In [ ]:
# Treatment and cell line mapping from TheseWells.txt
treatment_map = {
    # HCT116
    ('PB000139', 'C03'): ('HCT116', 'binimetinib'),
    ('PB000140', 'G05'): ('HCT116', 'binimetinib'),
    ('PB000139', 'D11'): ('HCT116', 'binimetinib'),
    ('PB000139', 'G08'): ('HCT116', 'abemaciclib'),
    ('PB000140', 'G08'): ('HCT116', 'abemaciclib'),
    ('PB000140', 'J04'): ('HCT116', 'abemaciclib'),
    ('PB000139', 'C23'): ('HCT116', 'SN-38'),
    ('PB000140', 'C23'): ('HCT116', 'SN-38'),
    ('PB000139', 'J06'): ('HCT116', 'SN-38'),
    ('PB000137', 'D12'): ('HCT116', 'DMSO'),
    ('PB000137', 'F06'): ('HCT116', 'DMSO'),
    ('PB000140', 'N05'): ('HCT116', 'DMSO'),
    # HT29
    ('PB000139', 'C14'): ('HT29', 'binimetinib'),
    ('PB000140', 'D22'): ('HT29', 'binimetinib'),
    ('PB000140', 'I13'): ('HT29', 'binimetinib'),
    ('PB000139', 'J15'): ('HT29', 'abemaciclib'),
    ('PB000140', 'B14'): ('HT29', 'abemaciclib'),
    ('PB000140', 'G19'): ('HT29', 'abemaciclib'),
    ('PB000139', 'D06'): ('HT29', 'SN-38'),
    ('PB000139', 'C12'): ('HT29', 'SN-38'),
    ('PB000140', 'J17'): ('HT29', 'SN-38'),
    ('PB000140', 'H23'): ('HT29', 'DMSO'),
    ('PB000141', 'G02'): ('HT29', 'DMSO'),
    ('PB000141', 'M23'): ('HT29', 'DMSO'),
}

def extract_mask(data):
    """Extract mask array from various formats."""
    if isinstance(data, np.ndarray):
        if data.dtype == object and data.shape == ():
            item = data.item()
            if isinstance(item, dict) and 'masks' in item:
                return item['masks']
        return data
    return data

def calculate_binary_iou(mask1, mask2):
    """Calculate IoU between two binary masks."""
    # Convert to binary
    binary1 = mask1 > 0
    binary2 = mask2 > 0
    
    intersection = np.logical_and(binary1, binary2).sum()
    union = np.logical_or(binary1, binary2).sum()
    
    if union == 0:
        return 0.0
    return intersection / union

def calculate_instance_iou(labels_pred, labels_gt):
    """Calculate per-object IoU: for each GT object, find best matching prediction.
    
    Returns mean IoU across all GT objects.
    """
    gt_ids = np.unique(labels_gt)
    gt_ids = gt_ids[gt_ids != 0]  # Remove background
    
    if len(gt_ids) == 0:
        return np.nan, 0
    
    pred_ids = np.unique(labels_pred)
    pred_ids = pred_ids[pred_ids != 0]
    
    if len(pred_ids) == 0:
        return 0.0, len(gt_ids)
    
    ious = []
    for gt_id in gt_ids:
        gt_mask = (labels_gt == gt_id)
        best_iou = 0.0
        
        for pred_id in pred_ids:
            pred_mask = (labels_pred == pred_id)
            intersection = np.logical_and(gt_mask, pred_mask).sum()
            union = np.logical_or(gt_mask, pred_mask).sum()
            if union > 0:
                iou = intersection / union
                if iou > best_iou:
                    best_iou = iou
        
        ious.append(best_iou)
    
    return np.mean(ious), len(gt_ids)

# Build results dataframe
results = []

# Get unique (barcode, seg_type, well, plane) combinations
keys_expert = {k[:4] for k in all_labels.keys() if k[4] == 'expert'}
keys_cellpose = {k[:4] for k in all_labels.keys() if k[4] == 'cellpose'}
common_keys = keys_expert & keys_cellpose

print(f"Found {len(common_keys)} matching expert-cellpose pairs")

# Check first pair to determine cellpose format
first_key = list(common_keys)[0]
sample_cellpose = extract_mask(all_labels[(*first_key, 'cellpose')])
cellpose_is_binary = sample_cellpose.dtype == bool or len(np.unique(sample_cellpose)) <= 2
print(f"Cellpose format: {'binary' if cellpose_is_binary else 'instance labels'}")

for barcode, seg_type, well, plane in common_keys:
    expert_mask = extract_mask(all_labels[(barcode, seg_type, well, plane, 'expert')])
    cellpose_mask = extract_mask(all_labels[(barcode, seg_type, well, plane, 'cellpose')])
    
    # Calculate binary IoU (overall segmentation agreement)
    binary_iou = calculate_binary_iou(cellpose_mask, expert_mask)
    
    # Calculate instance IoU only if cellpose has labels
    if not cellpose_is_binary:
        instance_iou, n_objects = calculate_instance_iou(cellpose_mask, expert_mask)
    else:
        instance_iou = np.nan
        n_objects = len(np.unique(expert_mask)) - 1  # Count from expert
    
    # Get treatment info
    cell_line, treatment = treatment_map.get((barcode, well), ('Unknown', 'Unknown'))
    
    # Depth in µm (assuming 5µm per plane)
    depth_um = int(plane) * 5
    
    results.append({
        'barcode': barcode,
        'well': well,
        'plane': int(plane),
        'depth_um': depth_um,
        'seg_type': seg_type,
        'cell_line': cell_line,
        'treatment': treatment,
        'binary_iou': binary_iou,
        'instance_iou': instance_iou,
        'n_objects': n_objects
    })

df_results = pd.DataFrame(results)
print(f"\nResults shape: {df_results.shape}")
print(f"\nBinary IoU stats:")
print(df_results['binary_iou'].describe())
df_results.head(10)

Panel A - IoU by depth (line plot with confidence bands):
Y-axis: IoU (0-1.0)
X-axis: Depth (µm) - 0, 5, 10, 15... 60
Lines: Separate for nuclei vs. cytoplasm channels
Shading: 95% CI or SD bands
Horizontal line: IoU = 0.7 threshold marker
Why: Shows the expected technical degradation you already discuss. Validates that depth affects segmentation predictably.

Panel D - Heatmap (IoU stratified by depth × treatment):
Rows: Depth bins (0-20µm, 20-40µm, 40-60µm)
Columns: Treatment groups
Color: Mean IoU (gradient: red=low, green=high)
Annotations: Cell count per bin
Why: Compact view of interaction between technical (depth) and biological (treatment) factors.


In [ ]:
# Panel A - IoU by depth (line plot with confidence bands)
fig, ax = plt.subplots(figsize=(10, 6))

for seg_type, color in [('nuclei', 'blue'), ('cell', 'orange')]:
    data = df_results[df_results['seg_type'] == seg_type]
    
    # Group by depth and calculate mean + CI
    grouped = data.groupby('depth_um')['binary_iou'].agg(['mean', 'std', 'count'])
    grouped['se'] = grouped['std'] / np.sqrt(grouped['count'])
    grouped['ci95'] = 1.96 * grouped['se']
    
    ax.plot(grouped.index, grouped['mean'], color=color, label=seg_type, linewidth=2)
    ax.fill_between(grouped.index, 
                    grouped['mean'] - grouped['ci95'],
                    grouped['mean'] + grouped['ci95'],
                    color=color, alpha=0.2)

# IoU = 0.5 threshold line
ax.axhline(y=0.5, color='red', linestyle='--', linewidth=1, label='IoU = 0.5 threshold')

ax.set_xlabel('Depth (µm)', fontsize=12)
ax.set_ylabel('IoU', fontsize=12)
ax.set_ylim(0, 1.0)
ax.set_title('Panel A: IoU by Depth', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Panel D - Heatmap (IoU stratified by depth × treatment)

# Create depth bins
df_results['depth_bin'] = pd.cut(df_results['depth_um'], 
                                  bins=[0, 20, 40, 60], 
                                  labels=['0-20µm', '20-40µm', '40-60µm'])

# Create pivot table for mean IoU
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, seg_type in enumerate(['nuclei', 'cell']):
    data = df_results[df_results['seg_type'] == seg_type]
    
    # Pivot for mean IoU
    pivot_iou = data.pivot_table(values='binary_iou', 
                                  index='depth_bin', 
                                  columns='treatment', 
                                  aggfunc='mean')
    
    # Pivot for cell counts (annotations)
    pivot_count = data.pivot_table(values='n_objects', 
                                    index='depth_bin', 
                                    columns='treatment', 
                                    aggfunc='sum').fillna(0).astype(int)
    
    # Create heatmap
    sns.heatmap(pivot_iou, 
                annot=pivot_count, 
                fmt='d',
                cmap='RdYlGn', 
                vmin=0, vmax=1,
                ax=axes[idx],
                cbar_kws={'label': 'Mean IoU'})
    
    axes[idx].set_title(f'{seg_type.capitalize()} - IoU by Depth × Perturbation', fontsize=12)
    axes[idx].set_xlabel('Perturbation')
    axes[idx].set_ylabel('Depth Bin')

plt.tight_layout()
plt.show()

# Summary statistics
print("\nSummary by depth bin and treatment:")
summary = df_results.groupby(['depth_bin', 'treatment', 'seg_type']).agg({
    'binary_iou': ['mean', 'std'],
    'n_objects': 'sum'
}).round(3)
print(summary)

# Save figure to svg
import matplotlib
matplotlib.rcParams['svg.fonttype'] = 'none'
save_panel(fig, 'SupplFig2d', data=df_results,
           caption='Cellpose vs expert-annotation IoU by depth and treatment',
           notebook='analysis/3_SupplFigure2/quantify_segmentation_error.ipynb')

In [ ]:
# Debug: Check mask structure first
sample_key = list(all_labels.keys())[0]
print(f"Sample key: {sample_key}")
sample_mask = all_labels[sample_key]
print(f"Type: {type(sample_mask)}")
print(f"Dtype: {sample_mask.dtype}")
print(f"Shape: {sample_mask.shape if hasattr(sample_mask, 'shape') else 'N/A'}")

# If it's an object array, check what's inside
if sample_mask.dtype == object:
    item = sample_mask.item()
    print(f"Item type: {type(item)}")
    if isinstance(item, dict):
        print(f"Dict keys: {item.keys()}")
        for k, v in item.items():
            if hasattr(v, 'shape'):
                print(f"  {k}: shape={v.shape}, dtype={v.dtype}")
            else:
                print(f"  {k}: {type(v)}")
else:
    print(f"Min: {sample_mask.min()}, Max: {sample_mask.max()}")
    print(f"Unique values: {len(np.unique(sample_mask))}")

In [ ]:
# Guarded: these are exploratory mask overlays, not a paper panel, and they need
# the 7.7 GB raw .ome.tiff set held outside the repo. Suppl 2d itself is produced
# above from the committed segmentation_iou_cached.csv, so a run without the raw
# images still yields the panel.
if (external('expert-annotation') / 'raw_images').is_dir():
    from skimage import io
    from skimage.segmentation import find_boundaries
    import matplotlib.patches as mpatches

    # Example image panel: 3 depths × 3 columns
    # Rows: shallow (5µm, plane 1), mid (25µm, plane 5), deep (45µm, plane 9)
    # Columns: raw image, expert outlines, agreement overlay (filled)

    raw_dir = external('expert-annotation') / 'raw_images'
    planes = [1, 5, 9]
    depth_labels = ['Shallow (5µm)', 'Mid (25µm)', 'Deep (45µm)']
    barcode = 'PB000137'
    well = 'D12'

    fig, axes = plt.subplots(3, 3, figsize=(15, 15))

    for row_idx, (plane, depth_label) in enumerate(zip(planes, depth_labels)):
        # Load raw image (HOECHST channel for nuclei)
        raw_path = raw_dir / f'Plate-{barcode}-Well-{well}-z{plane}-HOECHST.ome.tiff'
        raw_img = io.imread(raw_path)
    
        # Get masks
        expert_key = (barcode, 'nuclei', well, str(plane), 'expert')
        cellpose_key = (barcode, 'nuclei', well, str(plane), 'cellpose')
    
        expert_mask = extract_mask(all_labels[expert_key])
        cellpose_mask = extract_mask(all_labels[cellpose_key])
    
        # Find instance boundaries (preserves individual cell outlines)
        expert_boundaries = find_boundaries(expert_mask, mode='outer')
    
        # Find instance boundaries (preserves individual cell outlines)
        cellpose_boundaries = find_boundaries(cellpose_mask, mode='thick')
    
        # Binary masks for overlap calculation
        expert_binary = expert_mask > 0
        cellpose_binary = cellpose_mask > 0
    
        # Normalize raw image for display
        vmin, vmax = np.percentile(raw_img, [1, 99])
    
        # Column 1: Raw image
        axes[row_idx, 0].imshow(raw_img, cmap='gray', vmin=vmin, vmax=vmax)
        axes[row_idx, 0].set_title(f'{depth_label}\nRaw HOECHST')
        axes[row_idx, 0].axis('off')
    
        # Column 2: Raw + Expert outlines (cyan)
        axes[row_idx, 1].imshow(raw_img, cmap='gray', vmin=vmin, vmax=vmax)
        overlay_expert = np.zeros((*raw_img.shape, 4))
        overlay_expert[cellpose_boundaries] = [1, 0, 0, 1]  # Cyan outlines
        axes[row_idx, 1].imshow(overlay_expert)
        n_expert = len(np.unique(expert_mask)) - 1
        axes[row_idx, 1].set_title(f'Expert Outlines\n({n_expert} cells)')
        axes[row_idx, 1].axis('off')
    
        # Column 3: Filled agreement overlay
        axes[row_idx, 2].imshow(raw_img, cmap='gray', vmin=vmin, vmax=vmax)
    
        # Create filled overlay: green=agree, red=cellpose only, blue=expert only
        agreement = np.zeros((*raw_img.shape, 4))
        agreement[expert_binary & cellpose_binary] = [0, 1, 0, 0.5]    # Green: agreement
        agreement[cellpose_binary & ~expert_binary] = [1, 0, 0, 0.5]   # Red: cellpose only (FP)
        agreement[expert_binary & ~cellpose_binary] = [0, 0, 1, 0.5]   # Blue: expert only (FN)
    
        axes[row_idx, 2].imshow(agreement)
    
        # Calculate IoU for this plane
        iou = calculate_binary_iou(cellpose_mask, expert_mask)
        axes[row_idx, 2].set_title(f'Agreement (IoU={iou:.2f})\nGreen=agree, Red=CP only, Blue=Exp only')
        axes[row_idx, 2].axis('off')

    # Add legend
    legend_patches = [
        mpatches.Patch(color='green', label='Agreement'),
        mpatches.Patch(color='red', label='Cellpose only (FP)'),
        mpatches.Patch(color='blue', label='Expert only (FN)')
    ]
    fig.legend(handles=legend_patches, loc='lower center', ncol=3, fontsize=12, bbox_to_anchor=(0.5, 0.02))

    plt.suptitle(f'Segmentation Comparison: {barcode} Well {well} (Nuclei)', fontsize=14, y=0.98)
    plt.tight_layout(rect=[0, 0.05, 1, 0.96])
    plt.show()
else:
    print('SKIP mask overlays: raw_images not available (not a paper panel)')


In [ ]:
# Guarded: these are exploratory mask overlays, not a paper panel, and they need
# the 7.7 GB raw .ome.tiff set held outside the repo. Suppl 2d itself is produced
# above from the committed segmentation_iou_cached.csv, so a run without the raw
# images still yields the panel.
if (external('expert-annotation') / 'raw_images').is_dir():
    from skimage import io
    from skimage.segmentation import find_boundaries
    import matplotlib.patches as mpatches

    # Example image panel: 3 depths × 3 columns
    # Rows: shallow (5µm, plane 1), mid (25µm, plane 5), deep (45µm, plane 9)
    # Columns: raw image, expert outline, cellpose outline with agreement coloring

    raw_dir = external('expert-annotation') / 'raw_images'
    planes = [1, 5, 9]
    depth_labels = ['Shallow (5µm)', 'Mid (25µm)', 'Deep (45µm)']
    barcode = 'PB000137'
    well = 'D12'

    fig, axes = plt.subplots(3, 3, figsize=(15, 15))

    for row_idx, (plane, depth_label) in enumerate(zip(planes, depth_labels)):
        # Load raw image (PHAandWGA channel for cells)
        raw_path = raw_dir / f'Plate-{barcode}-Well-{well}-z{plane}-PHAandWGA.ome.tiff'
        raw_img = io.imread(raw_path)
    
        # Get masks
        expert_key = (barcode, 'cell', well, str(plane), 'expert')
        cellpose_key = (barcode, 'cell', well, str(plane), 'cellpose')
    
        expert_mask = extract_mask(all_labels[expert_key])
        cellpose_mask = extract_mask(all_labels[cellpose_key])
    
        # Create binary masks
        expert_binary = expert_mask > 0
        cellpose_binary = cellpose_mask > 0
    
        # Find boundaries using instance masks (not binary) to show individual cell outlines
        expert_boundary = find_boundaries(expert_mask, mode='outer')
        cellpose_boundary = find_boundaries(cellpose_mask, mode='outer')
        cellpose_boundary_vis = find_boundaries(cellpose_mask, mode='thick')

        # Normalize raw image for display
        vmin, vmax = np.percentile(raw_img, [1, 99])

        # Column 1: Raw image
        axes[row_idx, 0].imshow(raw_img, cmap='gray', vmin=vmin, vmax=vmax)
        axes[row_idx, 0].set_title(f'{depth_label}\nRaw PHAandWGA')
        axes[row_idx, 0].axis('off')
    
        # Column 2: Raw + CellPose instance outlines (red)
        axes[row_idx, 1].imshow(raw_img, cmap='gray', vmin=vmin, vmax=vmax)
        overlay_cellpose = np.zeros((*raw_img.shape, 4))
        overlay_cellpose[cellpose_boundary_vis] = [1, 0, 0, 1]
        axes[row_idx, 1].imshow(overlay_cellpose)
        n_cellpose = len(np.unique(cellpose_mask)) - 1
        axes[row_idx, 1].set_title(f'CellPose Annotation\n({n_cellpose} cells)')
        axes[row_idx, 1].axis('off')
    
        # Column 3: Agreement/disagreement overlay
        # Green = both agree, Red = cellpose only, Blue = expert only
        agreement = np.zeros((*raw_img.shape, 3))
        agreement[expert_binary & cellpose_binary] = [0, 1, 0]      # Green: agreement
        agreement[cellpose_binary & ~expert_binary] = [1, 0, 0]     # Red: cellpose only (false positive)
        agreement[expert_binary & ~cellpose_binary] = [0, 0, 1]     # Blue: expert only (false negative)
    
        axes[row_idx, 2].imshow(raw_img, cmap='gray', vmin=vmin, vmax=vmax)
        axes[row_idx, 2].imshow(agreement, alpha=0.5)
    
        # Calculate IoU for this plane
        iou = calculate_binary_iou(cellpose_mask, expert_mask)
        axes[row_idx, 2].set_title(f'Agreement (IoU={iou:.2f})\nGreen=agree, Red=CP only, Blue=Exp only')
        axes[row_idx, 2].axis('off')

    # Add legend
    legend_patches = [
        mpatches.Patch(color='green', label='Agreement'),
        mpatches.Patch(color='red', label='Cellpose only (FP)'),
        mpatches.Patch(color='blue', label='Expert only (FN)')
    ]
    fig.legend(handles=legend_patches, loc='lower center', ncol=3, fontsize=12, bbox_to_anchor=(0.5, 0.02))

    plt.suptitle(f'Segmentation Comparison: {barcode} Well {well} (Cells)', fontsize=14, y=0.98)
    plt.tight_layout(rect=[0, 0.05, 1, 0.96])
    plt.show()
else:
    print('SKIP mask overlays: raw_images not available (not a paper panel)')


#### Calculate error propegation 